# Evaluate DeNICE checkpoint on Google Colab

Notebook này mount Google Drive, clone code mới nhất, và evaluate đúng checkpoint `checkpoint_task_5_round_19.pt`. Checkpoint delta phải ở nguyên thư mục output cùng với `checkpoint_task_5_base.pt` và các round checkpoint trước đó. Nếu folder nằm trong **Shared with me/Được chia sẻ với tôi**, hãy tạo shortcut folder đó vào **My Drive** trước khi chạy Cell 2.


In [ ]:
# Cell 1 — mount Drive và clone repository mới nhất
from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')
REPO_PATH = Path('/content/FL_IL_IDS')
if REPO_PATH.exists():
    shutil.rmtree(REPO_PATH)
subprocess.run([
    'git', 'clone', '--depth', '1',
    'https://github.com/khoilv2005/FL_IL_IDS.git', str(REPO_PATH),
], check=True)
print('Repository:', subprocess.check_output(['git', '-C', str(REPO_PATH), 'rev-parse', '--short', 'HEAD'], text=True).strip())


In [ ]:
# Cell 2 — đặt đường dẫn data; checkpoint tự tìm theo tên chính xác
# DATA_DIR phải là thư mục 100-clients chứa dữ liệu client, không phải file checkpoint.
DRIVE_ROOT = Path('/content/drive/MyDrive')
DATA_DIR = Path('/content/drive/MyDrive/Anh Khôi ĐACN/Dataset/2023/federated_splits/100-clients')
CHECKPOINT_NAME = 'checkpoint_task_5_round_19.pt'
# Folder 100 clients phải được Add shortcut to Drive/My Drive trước khi dùng path này.
CHECKPOINT_PATH = Path('/content/drive/MyDrive/Anh Khôi ĐACN/Thực nghiệm/100 clients/DENICE22/checkpoint_task_5_round_19.pt')

if not DATA_DIR.is_dir():
    raise FileNotFoundError(f'DATA_DIR không tồn tại: {DATA_DIR}. Hãy sửa DATA_DIR ở Cell 2.')

if CHECKPOINT_PATH is None:
    candidates = sorted(DRIVE_ROOT.rglob(CHECKPOINT_NAME))
    if len(candidates) == 0:
        raise FileNotFoundError(f'Không tìm thấy {CHECKPOINT_NAME} trong {DRIVE_ROOT}.')
    if len(candidates) > 1:
        print('Tìm thấy nhiều checkpoint; hãy copy một đường dẫn bên dưới vào CHECKPOINT_PATH:')
        for candidate in candidates:
            print(candidate)
        raise RuntimeError('Checkpoint ambiguous — không tự chọn để tránh eval nhầm run.')
    CHECKPOINT_PATH = candidates[0]
else:
    CHECKPOINT_PATH = Path(CHECKPOINT_PATH)

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        f'Không thấy checkpoint: {CHECKPOINT_PATH}. Hãy tạo shortcut folder 100 clients vào My Drive hoặc sửa CHECKPOINT_PATH.'
    )

print('Dataset:', DATA_DIR)
print('Checkpoint:', CHECKPOINT_PATH)


In [ ]:
# Cell 3 — xác thực checkpoint task 5 / round 19 và chuỗi delta trước khi eval
import torch

header = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
assert header.get('algorithm') == 'denice', header.get('algorithm')
assert int(header.get('task_id', header.get('task', -1))) == 5, header.get('task_id')
assert int(header.get('round_id', header.get('final_round_id', -1))) == 19, header.get('round_id')

if header.get('checkpoint_type') == 'denice_delta_round':
    required = [CHECKPOINT_PATH.parent / header['base_path']]
    cursor = header
    while cursor.get('previous_round_path'):
        previous = CHECKPOINT_PATH.parent / cursor['previous_round_path']
        required.append(previous)
        cursor = torch.load(previous, map_location='cpu', weights_only=False)
    missing = [p for p in required if not p.is_file()]
    if missing:
        raise FileNotFoundError(f'Thiếu file delta/base: {missing}')
    print(f'Delta chain verified: {len(required)} dependency files.')
else:
    print('Full task-end checkpoint verified.')


In [ ]:
# Cell 4 — cumulative eval task 0-5, cùng mẫu cho hard/topk/nomask
EVAL_OUTPUT = DRIVE_ROOT / 'denice_task5_round19_eval.json'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
command = [
    sys.executable, str(REPO_PATH / 'eval_checkpoint.py'),
    '--checkpoint', str(CHECKPOINT_PATH),
    '--data-dir', str(DATA_DIR),
    '--device', device,
    '--router-mode', 'multiclass',
    '--evaluation-mode', 'representative',
    '--route-modes', 'hard,topk,nomask',
    '--eval-seed', '42',
    '--output', str(EVAL_OUTPUT),
]
print('Running:', ' '.join(command))
subprocess.run(command, check=True)
print(f'✓ Saved evaluation: {EVAL_OUTPUT}')


In [ ]:
# Cell 5 — xem kết quả
import json
with open(EVAL_OUTPUT, 'r', encoding='utf-8') as f:
    evaluation = json.load(f)
print(json.dumps(evaluation, indent=2))
